# 02 — CNN Baseline: ResNet-50

## Why ResNet-50?

ResNet-50 is the standard baseline for medical image classification:
- **Transfer learning**: ImageNet pretrained features transfer well to histopathology
- **Well-understood**: Easy to debug, benchmark, and compare against
- **Efficient**: 25.6M parameters, trains in minutes on a single GPU

This notebook trains the baseline and analyzes why it plateaus, motivating the move to Vision Transformers in Phase 2.

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from src.data.dataset import HistoDataset
from src.data.augmentations import get_train_transforms, get_val_transforms
from src.models.resnet_baseline import ResNet50Classifier
from src.training.losses import FocalLoss
from src.training.metrics import compute_macro_f1, compute_auroc, plot_confusion_matrix, MetricTracker

In [ ]:
# Load pre-trained checkpoint if available
output_dir = "../outputs/resnet/"
checkpoint_path = os.path.join(output_dir, "best_model.pt")

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    print(f"Loaded checkpoint from epoch {checkpoint['epoch']+1}")
    print(f"Best val macro-F1: {checkpoint['val_f1']:.4f}")
    print(f"Best val AUROC: {checkpoint['val_auroc']:.4f}")
else:
    print("No checkpoint found. Run training first:")
    print("  python scripts/train_resnet.py --config configs/resnet_config.yaml --data_dir data/ --output_dir outputs/resnet/ --device cpu")

In [ ]:
# Plot training curves
curves_path = os.path.join(output_dir, "training_curves.png")
if os.path.exists(curves_path):
    img = Image.open(curves_path)
    fig, ax = plt.subplots(figsize=(18, 5))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("Training Curves")
    plt.tight_layout()
    plt.show()
else:
    print("Training curves not found. Run training first.")

In [ ]:
# Display confusion matrix
cm_path = os.path.join(output_dir, "confusion_matrix.png")
if os.path.exists(cm_path):
    img = Image.open(cm_path)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("Test Set Confusion Matrix")
    plt.tight_layout()
    plt.show()
else:
    print("Confusion matrix not found. Run training first.")

## The Key Metric Pivot Story

### Why accuracy is misleading

Our dataset has 55% stroma tiles. A model that **always predicts stroma** achieves:
- **Accuracy ~ 55%** — looks mediocre but is actually the naive baseline
- **Macro-F1 ~ 0.18** — correctly reveals this model is useless for 3/4 classes

### The right metrics

| Metric | What it measures | Why we use it |
|--------|-----------------|---------------|
| **Macro-F1** | Average F1 across all classes equally | Primary metric — treats minority classes fairly |
| **AUROC** | Ranking quality across all thresholds | Threshold-independent comparison metric |
| **Accuracy** | % correct overall | Reported for context only — not used for decisions |

### Expected ResNet-50 results

On synthetic data with distinct class textures:
- **Accuracy**: ~87% — inflated by easy stroma classification
- **Macro-F1**: ~0.71 — reveals difficulty with tumor/immune discrimination
- **Tumor-Immune F1**: ~0.64 — the hardest pair (both have dark nuclei on pink background)

The tumor-immune confusion is the structural bottleneck that motivates Phase 2.

In [ ]:
# Demonstrate the majority-class baseline
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score

manifest = pd.read_csv("../data/manifest.csv")
test_set = manifest[manifest["split"] == "test"]
y_true = test_set["label"].values

# Always predict stroma
y_pred_majority = np.array(["stroma"] * len(y_true))
acc_majority = accuracy_score(y_true, y_pred_majority)
f1_majority = f1_score(y_true, y_pred_majority, average="macro", zero_division=0)

print("=== Majority-class baseline (always predict stroma) ===")
print(f"Accuracy:  {acc_majority:.2%}  <-- misleadingly high!")
print(f"Macro-F1:  {f1_majority:.4f}  <-- correctly low")
print()
print("This proves accuracy alone cannot evaluate imbalanced classification.")

## Analysis: Why did ResNet plateau?

ResNet-50's receptive field is the structural bottleneck:

1. **Limited receptive field**: ResNet-50's effective receptive field covers ~100px of the 224px input. It cannot attend to global spatial patterns.

2. **Local feature bias**: CNNs build features hierarchically from local to global. At 224px input, the deepest layers still have a relatively local view.

3. **No explicit global attention**: ResNet has no mechanism to relate distant parts of the image.

### What Phase 2 (ViT) addresses

Vision Transformers use self-attention to relate ALL patches from layer 1:
- **Global receptive field**: Every patch attends to every other patch
- **Explicit spatial reasoning**: Attention maps reveal which regions the model focuses on
- **Better minority class discrimination**: Global context helps distinguish tumor nests from immune clusters